# Predict the pedestal using EPEDNN-SC loop for SPARC baseline scenario

In [8]:
import os
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sys
from pathlib import Path
import tempfile
import urllib.request # needed for geqdsk import
ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(ROOT))
from src.profiles_loop_solve import profiles_loop_solve

tokamaker_python_path = os.getenv('OFT_ROOTPATH')
if tokamaker_python_path is not None:
    sys.path.append(os.path.join(tokamaker_python_path,'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh
from OpenFUSIONToolkit.TokaMaker.util import create_isoflux, read_eqdsk

In [7]:
# Import the SPARC baseline scenario geqdsk (freely available from SPARCPublic)

_SPARC_PRD = 'https://raw.githubusercontent.com/cfs-energy/SPARCPublic/main/PrimaryReferenceDischarge'

def fetch_text(filename):
    """Read a file from the SPARCPublic PRD folder over HTTP, without writing it to disk."""
    url = f'{_SPARC_PRD}/{urllib.parse.quote(filename)}'
    print(f'Fetching {filename} from SPARCPublic...', end=' ', flush=True)
    with urllib.request.urlopen(url) as resp:
        text = resp.read().decode()
    print('done.')
    return text


def read_eqdsk_from_text(text):
    """read_eqdsk() needs a path, so stage the gEQDSK in a temporary file and clean up."""
    with tempfile.NamedTemporaryFile('w', suffix='.geqdsk', delete=False) as tmp:
        tmp.write(text)
        tmp_path = tmp.name
    try:
        return read_eqdsk(tmp_path)
    finally:
        os.remove(tmp_path)


eqdsk = read_eqdsk_from_text(fetch_text('2 - SPARC_DN_PRD_freegs_20221013'))

print(f"  Ip      = {eqdsk['ip']/1.E6:.2f} MA")
print(f"  F0      = {eqdsk['rcentr']*eqdsk['bcentr']:.2f} T.m  (B0 = {eqdsk['rcentr']*eqdsk['bcentr']/1.85:.2f} T at R0 = 1.85 m)")
print(f"  p_axis  = {eqdsk['pres'][0]/1.E6:.2f} MPa")
print(f"  axis    = ({eqdsk['raxis']:.3f}, {eqdsk['zaxis']:.3f}) m")

Fetching 2 - SPARC_DN_PRD_freegs_20221013 from SPARCPublic... 

done.
  Ip      = 8.70 MA
  F0      = 22.49 T.m  (B0 = 12.16 T at R0 = 1.85 m)
  p_axis  = 2.60 MPa
  axis    = (1.890, -0.000) m


In [ ]:
# Parameters for model #

# Output directory
out_dir = 'SPARC_output_logs'

# Scan parameters
x_res = 20
free_params = {
    'alpha_crit': 0.1,
    'C_KBM': 0.1,
    'De_chie_etg': 0.1,
    'nFC_x0': 3.16228e15,
    'ncx_x0_ratio': 1.259
}
eped_tol_max = 1e-5
eped_iter_max = 1000
EPEDNN_core = 'previous T, stiched ne'
kbm_treatment = "picard"
kbm_gate_eps = 0.1
picard_gate_mode = "average"
picard_relax = 1.0

verbose = False

In [5]:
# Model run
ped_wid, ped_h_out, gfile_pres, gfile_pres_grid = profiles_loop_solve(
    MHD_FP = mhd_fp,
    kprof_loc = 'manual rho grid',
    manual_profs = manual_profs,
    P_tot_e = ( 21.5 + 0.8 + 227 ) * 0.5 * 1e6, # table 4 of Hillesheim et al. 2026
    out_dir = out_dir,
    x_res = x_res,
    free_params = free_params,
    eped_tol_max = eped_tol_max,
    eped_iter_max = eped_iter_max,
    kbm_gate_eps = kbm_gate_eps,
    EPEDNN_core = EPEDNN_core,
    kbm_treatment = kbm_treatment,
    picard_gate_mode = picard_gate_mode,
    picard_relax = picard_relax,
    ig = 'manual',
    epednn_model = 'EPED_SPARC',
    verbose = verbose,
)

Setting up EPEDNN...


I0000 00:00:1785791136.600456 1719726 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785791136.601916 1719726 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785791136.649601 1719726 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785791138.069905 1719726 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

bt: [11.34364597]
Base model built.
EPEDNN-SC Loop Iter 0
betan: [1.21694598]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step
[-4.8060525e+08  1.2156768e+05]
Pedestal height: -480605248.0 MPa, Pedestal width: 121567.6796875 (psi_N)


/mnt/homes_global/jal2351/software/saarelma-conner-ped/src/solver.py:179: RuntimeWarning: invalid value encountered in sqrt
  self.V_th_i = np.sqrt(2*k_B*self.T_i_K/(M_i*self.M_eff)) # m/s, per psi_N_eval for Ti
/mnt/homes_global/jal2351/software/saarelma-conner-ped/src/solver.py:180: RuntimeWarning: invalid value encountered in sqrt
  self.V_th_e = np.sqrt(2*k_B*self.T_e_K/M_e) # m/s, per psi_N_eval for Te
/mnt/homes_global/jal2351/software/saarelma-conner-ped/src/solver.py:182: RuntimeWarning: invalid value encountered in sqrt
  self.V_cx = np.sqrt(2*k_B*self.T_i_K/(np.pi * M_i*self.M_eff)) # m/s, per psi_N_eval for Ti
/mnt/homes_global/jal2351/software/saarelma-conner-ped/src/adas/adas_ionisation.py:89: RuntimeWarning: invalid value encountered in log10
  lTe    = np.clip(np.log10(np.atleast_1d(Te_eV).ravel()),  Te_lo, Te_hi)
/mnt/homes_global/jal2351/software/saarelma-conner-ped/src/solver.py:191: RuntimeWarning: invalid value encountered in sqrt
  self.c_s = (self.e_i * self.T_e *

ValueError: cannot reshape array of size 0 into shape (0,newaxis)

In [ ]:
# Model output #

def pres_pred(ped_wid, gfile_pres, gfile_pres_grid):
    """
    Calculate pedestal pressure from pedestal width and MHD equilibrium file.
    """
    psiN_top = 1 - ped_wid
    return np.interp(psiN_top, gfile_pres_grid, gfile_pres)


p_gfile = pres_pred(ped_wid, gfile_pres, gfile_pres_grid)
out_dict = {
    'mhd_fp': mhd_fp,
    'profiles': manual_profs,
    'ped_h': ped_h_out,
    'ped_wid': ped_wid,
    'p_gfile': p_gfile,
}
np.save(output_dir+'/ARC_pedestal_prediction.npy', out_dict)